# Задача 6.16

**Задача.** Дана первая квадратичная форма поверхности. Найти уравнение линий, которые в каждой своей точке делят пополам углы между координатными линиями поверхности.

Так как явная поверхность не задана, основной рисунок делается в плоскости параметров $(u,v)$: поле направлений и интегральные кривые биссектрис.

In [ ]:
# Базовые библиотеки для аналитики и визуализации
import numpy as np
import matplotlib.pyplot as plt
from math import sin, cos, tan, sqrt, pi

try:
    import sympy as sp
except ImportError:
    sp = None

EPS = 1e-9
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
def setup_2d(xlim=(-5, 5), ylim=(-5, 5), title=None, xlabel='x', ylabel='y'):
    """Создает 2D-плоскость с осями координат и равным масштабом."""
    fig, ax = plt.subplots()
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    return fig, ax


def plot_points_2d(ax, points, labels=None):
    """Рисует точки на 2D-графике."""
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], s=45)
        if label:
            ax.text(p[0], p[1], '  ' + label)


def plot_parametric_2d(ax, xy_func, t_range, n=800, label=None):
    """Рисует плоскую параметрическую кривую t -> (x(t), y(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    xy = np.asarray(xy_func(t), dtype=float)
    ax.plot(xy[0], xy[1], label=label)
    if label:
        ax.legend()
    return xy


def plot_implicit_2d(ax, F, xlim, ylim, n=500, level=0, label=None):
    """Рисует неявную кривую F(x,y)=level через contour."""
    xs = np.linspace(xlim[0], xlim[1], n)
    ys = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(xs, ys)
    Z = F(X, Y)
    cs = ax.contour(X, Y, Z, levels=[level])
    if label:
        cs.collections[0].set_label(label)
        ax.legend()
    return cs

## Формула для наклонов биссектрис

In [ ]:
def bisector_slopes(E, F, G):
    """
    Для ds^2 = E du^2 + 2F du dv + G dv^2 возвращает два наклона p=dv/du.
    Условие: угол между (1,p) и e_u равен +/- углу между (1,p) и e_v.
    """
    A = np.sqrt(E)
    B = np.sqrt(G)
    denom_plus = B*F - A*G
    denom_minus = B*F + A*G
    p_plus = (A*F - B*E) / denom_plus
    p_minus = (-A*F - B*E) / denom_minus
    return p_plus, p_minus


def metric_case(case, u, v):
    if case == 'a':
        E = 1 + np.exp(2*v)
        F = -np.exp(2*u)
        G = np.exp(2*u)
    elif case == 'b':
        E = 1 + 0*u
        F = (2*u - 1) / 2
        G = 1 + u**2
    elif case == 'c':
        E = 1 + 0*u
        F = np.sin(u - v)
        G = 1 + 0*u
    elif case == 'd':
        E = 1 + (1 + v)**2
        F = -(1 + v)**2
        G = (1 + v)**2
    else:
        raise ValueError('case must be a,b,c,d')
    return E, F, G

## Визуализация поля направлений в плоскости параметров

In [ ]:
case = 'c'  # поменяй на 'a', 'b', 'c', 'd'

u_grid = np.linspace(-2, 2, 25)
v_grid = np.linspace(-2, 2, 25)
U, V = np.meshgrid(u_grid, v_grid)
E, F, G = metric_case(case, U, V)
P1, P2 = bisector_slopes(E, F, G)

fig, ax = setup_2d((-2, 2), (-2, 2), title=f'6.16{case}: направления биссектрис', xlabel='u', ylabel='v')
# Вектор направления: (du,dv)=(1,p). Нормируем для аккуратной картинки.
for P, label in [(P1, 'family 1'), (P2, 'family 2')]:
    DU = np.ones_like(P)
    DV = P
    L = np.sqrt(DU**2 + DV**2)
    ax.quiver(U, V, DU/L, DV/L, angles='xy', scale_units='xy', scale=12, alpha=0.65, label=label)
ax.legend()
plt.show()

## Заготовка для интегральных кривых

In [ ]:
def rk4_integrate_slope(f, u0, v0, u_span, h=0.02):
    """Интегрирует dv/du=f(u,v) методом RK4."""
    u_start, u_end = u_span
    direction = 1 if u_end >= u_start else -1
    h = abs(h) * direction
    us = [u_start]
    vs = [v0]
    u, v = u_start, v0
    while (u - u_end) * direction < 0:
        if abs(u + h - u_start) > abs(u_end - u_start):
            h = u_end - u
        k1 = f(u, v)
        k2 = f(u + h/2, v + h*k1/2)
        k3 = f(u + h/2, v + h*k2/2)
        k4 = f(u + h, v + h*k3)
        v = v + h*(k1 + 2*k2 + 2*k3 + k4)/6
        u = u + h
        us.append(u)
        vs.append(v)
    return np.array(us), np.array(vs)


def slope_functions_for_case(case):
    def p1(u, v):
        E, F, G = metric_case(case, u, v)
        return bisector_slopes(E, F, G)[0]
    def p2(u, v):
        E, F, G = metric_case(case, u, v)
        return bisector_slopes(E, F, G)[1]
    return p1, p2

case = 'c'
p1, p2 = slope_functions_for_case(case)
fig, ax = setup_2d((-2, 2), (-3, 3), title=f'6.16{case}: интегральные кривые биссектрис', xlabel='u', ylabel='v')

for v_start in np.linspace(-2, 2, 5):
    us, vs = rk4_integrate_slope(p1, 0, v_start, (-2, 2), h=0.02)
    ax.plot(us, vs)
    us, vs = rk4_integrate_slope(p2, 0, v_start, (-2, 2), h=0.02)
    ax.plot(us, vs, linestyle='--')
plt.show()

## Место для аналитической записи

После получения $p_1(u,v)$ и $p_2(u,v)$ нужно решить два ОДУ:

$$\frac{dv}{du}=p_1(u,v),\qquad \frac{dv}{du}=p_2(u,v).$$

Для финального решения лучше вывести формулы вручную/через SymPy и записать семейства интегральных кривых.